In [ ]:
import pandas as pd
import tqdm, os, glob
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [ ]:
token_df

In [6]:
for subject in token_df["subject"].unique():
    os.makedirs(f"aemri75anno_scalers", exist_ok=True)
    if os.path.exists(f"aemri75anno_scalers/{subject}.pkl"):
        continue

    subject_token_df = token_df[token_df["subject"]==subject]

    data = {}

    for start, end, art_f in tqdm.tqdm(zip(subject_token_df["start"], subject_token_df["end"], subject_token_df["art_f"]), total=len(subject_token_df)):
        features = get_aemri75anno_articulatory_features(art_f, start, end)
        if features is None:
            continue
        features = features.reshape(features.shape[0], -1)
        for idx in range(features.shape[1]):
            if idx not in data:
                data[idx] = []
            data[idx].append(features[:, idx])

    scalers = {}
    for idx in tqdm.tqdm(data, total=len(data)):
        data[idx] = np.concatenate(data[idx]).reshape(-1, 1)
        scaler = StandardScaler()
        scaler.fit(data[idx])
        scalers[idx] = scaler

    with open(f"aemri75anno_scalers/{subject}.pkl", "wb") as f:
        pickle.dump(scalers, f)    

In [7]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"aemri75anno_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder

In [9]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [11]:
log = {
    "batch_idx": [],
    "loss": [],
}
art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=1024,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aemri75anno_articulatory_features(art_f, start, end, scalers=scalers[subject])

    if articulatory_features is None:
        continue

    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 174, 2)

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_x = art_model(art_batch)
    art_loss, art_static, art_delta = combined_loss(
        art_out, art_batch
    )

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_x)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aemri75anno/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75anno/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aemri75anno/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75anno/art_model_training_log.csv", index=False)

300: ART 0.5410:   3%|▎         | 1209/37849 [02:40<1:25:08,  7.17it/s]

[ART] latent var mean=3.1181e+02, min=4.3919e-03
[ART] input std=8.4275e-01, recon std=3.3168e-01


600: ART 0.4851:   6%|▋         | 2412/37849 [05:17<1:14:55,  7.88it/s]

[ART] latent var mean=6.4805e+02, min=1.7424e-03
[ART] input std=7.5222e-01, recon std=3.4188e-01


900: ART 0.6598:  10%|▉         | 3617/37849 [08:00<1:16:30,  7.46it/s]

...

9000: ART 0.3287:  96%|█████████▌| 36170/37849 [1:23:08<03:41,  7.59it/s]

[ART] latent var mean=5.5032e+04, min=3.3267e+00
[ART] input std=8.0248e-01, recon std=3.5903e-01


9300: ART 0.2593:  99%|█████████▊| 37375/37849 [1:25:55<01:13,  6.48it/s]

[ART] latent var mean=3.3704e+03, min=2.5822e-02
[ART] input std=7.2962e-01, recon std=2.9774e-01


9417: ART 0.4695: 100%|██████████| 37849/37849 [1:27:03<00:00,  7.25it/s]


In [12]:
BATCH_SIZE = 4

In [14]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    try:
        mfccs = get_mfccs(ac_f, start, end).T
    except Exception as e:
        continue

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri75anno/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri75anno/ac_model_training_log.csv", index=False)

999: AC 123.5059:  11%|█         | 3997/37849 [00:23<04:12, 134.28it/s]

[AC ] latent var mean=6.2394e+01, min=3.3530e-03
[AC ] input std=1.2330e+01, recon std=3.4787e+00


1999: AC 152.5132:  21%|██        | 8010/37849 [00:54<03:54, 127.30it/s]

[AC ] latent var mean=1.0780e+02, min=2.5715e-03
[AC ] input std=1.1151e+01, recon std=3.7269e+00


2999: AC 141.4753:  32%|███▏      | 12019/37849 [01:25<03:15, 132.04it/s]

...

7999: AC 170.9995:  85%|████████▍ | 32106/37849 [03:57<00:53, 106.54it/s]

[AC ] latent var mean=3.3586e+02, min=1.3021e-02
[AC ] input std=1.3013e+01, recon std=2.1101e+00


8999: AC 153.4428:  95%|█████████▌| 36126/37849 [04:42<00:18, 95.70it/s] 

[AC ] latent var mean=5.0422e+02, min=5.1817e-05
[AC ] input std=1.3050e+01, recon std=2.4203e+00


9426: AC 89.2166: 100%|██████████| 37849/37849 [05:04<00:00, 124.11it/s] 
